In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)




In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# 3. Create DataLoaders

from torch.utils.data import DataLoader, TensorDataset


train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
first_sample, _ = train_dataset[0]
print(f"Shape of one sample: {first_sample.shape}")


In [ ]:
# 5. Display sample images
from torchvision.datasets import MNIST
from torchvision.transforms.functional import to_tensor

# Training dataset
train_dataset = MNIST(
    root='./datasets',     # Dataset storage path
    train=True,            # Use training data
    transform=to_tensor,   # Convert images to tensors
    download=True          # Download if not available
)

# Testing dataset
test_dataset = MNIST(
    root='./datasets',     # Dataset storage path
    train=False,           # Use test data
    transform=to_tensor,   # Convert images to tensors
    download=True          # Download if not available
)

# print one sample from the dataset
# Each sample consists of an image tensor and its label
sample_image, sample_label = train_dataset[0]

print(f"\n Image shape: {sample_image.shape}")  # (1, 28, 28)
print(f"Label: {sample_label}")


In [ ]:
# Task 1: Write your model class here:
import torch
import torch.nn as nn

linear = nn.Linear(in_features=3, out_features=1)

x = torch.randn(4, 3)

y = linear(x)

print("Input shape:", x.shape)
print("Output shape:", y.shape)

In [ ]:
# Task 2: Write your training loop here:



In [ ]:
# Task 3: Write your validation loop here:


In [ ]:
# Task 4: Define device, model, loss, optimizer:
import torch.nn.functional as F
class TabularNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(TabularNN, self).__init__()
        # Layer 1: Input to Hidden
        self.fc1 = nn.Linear(input_size, hidden_size)
        # Layer 2: Hidden to Hidden
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        # Layer 3: Hidden to Output
        self.fc3 = nn.Linear(hidden_size, num_classes)

        self.dropout = nn.Dropout(0.2) # Optional: prevents overfitting

    def forward(self, x):
        x = F.relu(self.fc1(x))   # Activation 1
        x = self.dropout(x)       # Apply dropout
        x = F.relu(self.fc2(x))   # Activation 2
        # NOTE: No Softmax here if using CrossEntropyLoss!
        x = self.fc3(x)
        return x

import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TabularNN(input_size=X_train.shape[1], hidden_size=64, num_classes=len(y)).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:
epochs = 20
for epoch in range(epochs):
    model.train() # Set to training mode
    train_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Standard 5-step process:
        optimizer.zero_grad()           # 1. Reset gradients
        outputs = model(batch_X)        # 2. Forward pass
        loss = criterion(outputs, batch_y) # 3. Calculate loss
        loss.backward()                 # 4. Backward pass
        optimizer.step()                # 5. Update weights

        train_loss += loss.item()

model.eval() # Set to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient tracking (saves memory)
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs, 1) # Get class with highest logit
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

accuracy = 100 * correct / total
print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Acc: {accuracy:.2f}%")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 5))

plt.plot(train_loss, label='Train Loss')
plt.plot(val_loss, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: